# RAG Testing Notebook

This notebook allows you to:
1. **Test Retrieval**: Query the vector database to see what chunks are retrieved
2. **Test Generation**: Send retrieved context to different LLM providers (OpenRouter, Google, etc.)

**Note**: This is for testing only - does not modify any production code.

## Setup and Configuration

In [1]:
import psycopg2
from psycopg2.extras import RealDictCursor
from sentence_transformers import SentenceTransformer
import numpy as np
import json
from typing import List, Dict

# For LLM testing
import requests
from IPython.display import display, Markdown, HTML

print("✅ Libraries imported successfully")

C:\Users\gensh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully


## Configuration

In [7]:
# ============================================
# DATABASE CONFIGURATION
# ============================================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "chatbot_itl",
    "user": "postgres",
    "password": "123456"
}

# ============================================
# RAG CONFIGURATION
# ============================================
TENANT_ID = "1193a40f-1d03-4ecd-a601-901a55589f56"  # Change this to test different tenants
COLLECTION_NAME = "knowledge_documents"
TOP_K = 5  # Number of chunks to retrieve

# ============================================
# TEST QUERY
# ============================================
TEST_QUERY = "Hướng dẫn tạo mới bảng báo giá bán LCL?"

print(f"✅ Configuration set:")
print(f"   Tenant ID: {TENANT_ID}")
print(f"   Collection: {COLLECTION_NAME}")
print(f"   Top-K: {TOP_K}")
print(f"   Query: {TEST_QUERY}")

✅ Configuration set:
   Tenant ID: 1193a40f-1d03-4ecd-a601-901a55589f56
   Collection: knowledge_documents
   Top-K: 5
   Query: Hướng dẫn tạo mới bảng báo giá bán LCL?


## Part 1: Test Retrieval

This section queries the vector database and shows what chunks are retrieved.

In [8]:
# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded: all-MiniLM-L6-v2 (384 dimensions)")

Loading embedding model...
✅ Embedding model loaded: all-MiniLM-L6-v2 (384 dimensions)


In [9]:
def retrieve_chunks(query: str, tenant_id: str, top_k: int = 3) -> List[Dict]:
    """
    Retrieve relevant chunks from the vector database.
    
    Returns:
        List of dicts with keys: content, distance, metadata
    """
    # Connect to database
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    
    # Get collection ID
    cur.execute("""
        SELECT uuid FROM langchain_pg_collection WHERE name = %s
    """, (COLLECTION_NAME,))
    collection = cur.fetchone()
    
    if not collection:
        raise ValueError(f"Collection '{COLLECTION_NAME}' not found")
    
    collection_id = collection['uuid']
    
    # Generate query embedding
    query_embedding = embedder.encode(query).tolist()
    
    # Search for similar chunks
    cur.execute("""
        SELECT
            document as content,
            cmetadata as metadata,
            embedding <=> %s::vector as distance
        FROM langchain_pg_embedding
        WHERE collection_id = %s
        AND cmetadata->>'tenant_id' = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_embedding, collection_id, tenant_id, query_embedding, top_k))
    
    results = cur.fetchall()
    
    cur.close()
    conn.close()
    
    return results

print("✅ Retrieval function defined")

✅ Retrieval function defined


In [5]:
# Execute retrieval
print(f"\n{'='*100}")
print(f"RETRIEVAL TEST")
print(f"{'='*100}")
print(f"Query: {TEST_QUERY}")
print(f"Top-K: {TOP_K}")
print()

retrieved_chunks = retrieve_chunks(TEST_QUERY, TENANT_ID, TOP_K)

print(f"✅ Retrieved {len(retrieved_chunks)} chunks\n")

# Display each chunk
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{'='*100}")
    print(f"CHUNK {i}/{len(retrieved_chunks)}")
    print(f"{'='*100}")
    print(f"Distance (cosine): {chunk['distance']:.4f}")
    print(f"Section: {chunk['metadata'].get('section_title', 'Unknown')}")
    print(f"Page: {chunk['metadata'].get('page_number', 'Unknown')}")
    print(f"Length: {len(chunk['content'])} characters")
    print(f"\nContent:")
    print(f"-" * 100)
    print(chunk['content'])
    print()


RETRIEVAL TEST
Query: Hướng dẫn tạo mới bảng báo giá bán LCL?
Top-K: 3

✅ Retrieved 3 chunks

CHUNK 1/3
Distance (cosine): 0.1111
Section: 4.5.2. Bảng giá LCL
Page: 158
Length: 43 characters

Content:
----------------------------------------------------------------------------------------------------
Hình ảnh : Tạo mới bảng báo giá bán LCL (2)

CHUNK 2/3
Distance (cosine): 0.1155
Section: 4.5.2. Bảng giá LCL
Page: 159
Length: 43 characters

Content:
----------------------------------------------------------------------------------------------------
Hình ảnh : Tạo mới bảng báo giá bán LCL (3)

CHUNK 3/3
Distance (cosine): 0.1170
Section: 4.5.2. Bảng giá LCL
Page: 157
Length: 43 characters

Content:
----------------------------------------------------------------------------------------------------
Hình ảnh : Tạo mới bảng báo giá bán LCL (1)



### Combine Retrieved Context

In [6]:
# Combine all retrieved chunks into context
context_parts = []
for i, chunk in enumerate(retrieved_chunks, 1):
    section = chunk['metadata'].get('section_title', 'Unknown')
    context_parts.append(f"[Document {i} - Section: {section}]\n{chunk['content']}")

combined_context = "\n\n".join(context_parts)

print(f"{'='*100}")
print(f"COMBINED CONTEXT (to send to LLM)")
print(f"{'='*100}")
print(f"Total length: {len(combined_context)} characters")
print(f"Number of chunks: {len(retrieved_chunks)}")
print()
print(combined_context)

COMBINED CONTEXT (to send to LLM)
Total length: 265 characters
Number of chunks: 3

[Document 1 - Section: 4.5.2. Bảng giá LCL]
Hình ảnh : Tạo mới bảng báo giá bán LCL (2)

[Document 2 - Section: 4.5.2. Bảng giá LCL]
Hình ảnh : Tạo mới bảng báo giá bán LCL (3)

[Document 3 - Section: 4.5.2. Bảng giá LCL]
Hình ảnh : Tạo mới bảng báo giá bán LCL (1)


## Part 2: Test LLM Generation

This section sends the retrieved context to different LLM providers for testing.

### LLM Configuration

Choose one of the providers below and configure your API key.

In [ ]:
# ============================================
# OPTION 1: OpenRouter
# ============================================
OPENROUTER_API_KEY = ""  # Enter your OpenRouter API key
OPENROUTER_MODEL = "anthropic/claude-3.5-sonnet"  # or "openai/gpt-4", etc.

# ============================================
# OPTION 2: Google AI (Gemini)
# ============================================
GOOGLE_API_KEY = ""  # Enter your Google AI API key
GOOGLE_MODEL = "gemini-1.5-pro"  # or "gemini-1.5-flash"

# ============================================
# OPTION 3: OpenAI
# ============================================
OPENAI_API_KEY = ""  # Enter your OpenAI API key
OPENAI_MODEL = "gpt-4-turbo"  # or "gpt-3.5-turbo"

# Choose which provider to use
USE_PROVIDER = "openrouter"  # Options: "openrouter", "google", "openai"

print(f"✅ LLM Provider selected: {USE_PROVIDER.upper()}")

In [ ]:
def call_openrouter(query: str, context: str, api_key: str, model: str) -> str:
    """
    Call OpenRouter API
    """
    url = "https://openrouter.ai/api/v1/chat/completions"
    
    prompt = f"""Dựa trên thông tin sau đây, hãy trả lời câu hỏi của người dùng một cách chi tiết và chính xác.

Thông tin tham khảo:
{context}

Câu hỏi: {query}

Hãy trả lời câu hỏi dựa trên thông tin đã cung cấp. Nếu thông tin không đủ để trả lời, hãy nói rõ điều đó."""
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['choices'][0]['message']['content']


def call_google_ai(query: str, context: str, api_key: str, model: str) -> str:
    """
    Call Google AI API (Gemini)
    """
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    
    prompt = f"""Dựa trên thông tin sau đây, hãy trả lời câu hỏi của người dùng một cách chi tiết và chính xác.

Thông tin tham khảo:
{context}

Câu hỏi: {query}

Hãy trả lời câu hỏi dựa trên thông tin đã cung cấp. Nếu thông tin không đủ để trả lời, hãy nói rõ điều đó."""
    
    headers = {
        "Content-Type": "application/json"
    }
    
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['candidates'][0]['content']['parts'][0]['text']


def call_openai(query: str, context: str, api_key: str, model: str) -> str:
    """
    Call OpenAI API
    """
    url = "https://api.openai.com/v1/chat/completions"
    
    prompt = f"""Dựa trên thông tin sau đây, hãy trả lời câu hỏi của người dùng một cách chi tiết và chính xác.

Thông tin tham khảo:
{context}

Câu hỏi: {query}

Hãy trả lời câu hỏi dựa trên thông tin đã cung cấp. Nếu thông tin không đủ để trả lời, hãy nói rõ điều đó."""
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    
    result = response.json()
    return result['choices'][0]['message']['content']


print("✅ LLM calling functions defined")

In [ ]:
# Call the selected LLM
print(f"\n{'='*100}")
print(f"LLM GENERATION TEST")
print(f"{'='*100}")
print(f"Provider: {USE_PROVIDER.upper()}")
print(f"Query: {TEST_QUERY}")
print(f"Context length: {len(combined_context)} characters")
print()

try:
    if USE_PROVIDER == "openrouter":
        if not OPENROUTER_API_KEY:
            raise ValueError("Please set OPENROUTER_API_KEY")
        print(f"Model: {OPENROUTER_MODEL}")
        print("Calling OpenRouter API...\n")
        response = call_openrouter(TEST_QUERY, combined_context, OPENROUTER_API_KEY, OPENROUTER_MODEL)
        
    elif USE_PROVIDER == "google":
        if not GOOGLE_API_KEY:
            raise ValueError("Please set GOOGLE_API_KEY")
        print(f"Model: {GOOGLE_MODEL}")
        print("Calling Google AI API...\n")
        response = call_google_ai(TEST_QUERY, combined_context, GOOGLE_API_KEY, GOOGLE_MODEL)
        
    elif USE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise ValueError("Please set OPENAI_API_KEY")
        print(f"Model: {OPENAI_MODEL}")
        print("Calling OpenAI API...\n")
        response = call_openai(TEST_QUERY, combined_context, OPENAI_API_KEY, OPENAI_MODEL)
        
    else:
        raise ValueError(f"Unknown provider: {USE_PROVIDER}")
    
    print(f"{'='*100}")
    print(f"LLM RESPONSE")
    print(f"{'='*100}")
    print(response)
    print()
    
    # Display as markdown for better formatting
    display(Markdown(response))
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

## Summary

This notebook allows you to:
1. Test different queries and top-k values
2. See exactly what chunks are retrieved from the database
3. Test different LLM providers without modifying production code

### To modify test parameters:
- Change `TENANT_ID` to test different tenants
- Change `TOP_K` to retrieve more/fewer chunks
- Change `TEST_QUERY` to test different questions
- Change `USE_PROVIDER` and configure API keys to test different LLMs